## PRACTICA OBLIGATORIA: **Arboles de Decisión y Ajuste de Hiperparámetros**

* La práctica obligatoria de esta unidad consiste en encontrar el mejor modelo para resolver un problema de predicción de si los destinatarios de una campaña de marketing adquirirán un producto concreto.
* Recuerda que debes subirla a tu repositorio personal antes de la sesión en vivo para que puntúe adecuadamente.
* Recuerda también que no es necesario que esté perfecta, sólo es necesario que se vea el esfuerzo.
* Esta práctica se resolverá en la sesión en vivo correspondiente y la solución se publicará en el repo del curso.

### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import warnings
warnings.filterwarnings('ignore')
print("Librerías cargadas correctamente")

### Descripción

En el directorio data encontrarás un dataset con datos de campañas de marketing directo de una institución bancaria portuguesa (Bank Marketing - UCI). El objetivo es predecir si el cliente se suscribirá a un depósito a plazo (`y`).

Las variables se dividen en datos del cliente, datos del último contacto y atributos adicionales de campaña.

### Carga y exploración inicial

In [ ]:
# Carga (separador ';')
df = pd.read_csv('data/bank-full.csv', sep=';')
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### Identificación del target y análisis de su distribución

In [ ]:
# Distribución del target 'y'
print("Distribución absoluta:")
print(df['y'].value_counts())
print()
print("Distribución relativa:")
print(df['y'].value_counts(normalize=True).round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['y'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'], edgecolor='black')
axes[0].set_title('Distribución target (absoluta)')
axes[0].set_xlabel('Suscripción depósito')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(rotation=0)

df['y'].value_counts(normalize=True).plot(kind='bar', ax=axes[1], color=['steelblue','coral'], edgecolor='black')
axes[1].set_title('Distribución target (%)')
axes[1].set_xlabel('Suscripción depósito')
axes[1].tick_params(rotation=0)
plt.tight_layout()
plt.show()

print('''
Observación: Dataset muy desequilibrado. Solo ~11.7% de clientes suscribieron el depósito.
Se usará F1-score como métrica principal y class_weight=balanced en el árbol.
''')

### Preprocesamiento

In [ ]:
# Identificar columnas categóricas (excluir target)
cat_cols = [c for c in df.select_dtypes(include='object').columns if c != 'y']
print("Columnas categóricas:", cat_cols)

# Codificación con LabelEncoder
df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

# Target binario
df_enc['y_bin'] = (df_enc['y'] == 'yes').astype(int)
df_enc.drop('y', axis=1, inplace=True)

print(f"Columnas finales: {df_enc.columns.tolist()}")
df_enc.head()

### División Train/Test

In [ ]:
X = df_enc.drop('y_bin', axis=1)
y = df_enc['y_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} muestras  | positivos: {y_train.mean():.2%}")
print(f"Test:  {X_test.shape[0]} muestras   | positivos: {y_test.mean():.2%}")

### Mini-EDA sobre train

In [ ]:
corr = X_train.corrwith(y_train).abs().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
corr.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Correlación absoluta de features con el target (train)')
plt.ylabel('|Correlación|')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Top 5 features más correladas con el target:")
print(corr.head())

### Modelo Baseline: Regresión Logística (hiperparámetros por defecto)

In [ ]:
# Escalado necesario para LR
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr_baseline = LogisticRegression(random_state=42, max_iter=1000)
lr_cv = cross_val_score(lr_baseline, X_train_sc, y_train, cv=5, scoring='f1')
print(f"Regresión Logística (baseline) - F1 CV: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")

### Árbol de Decisión: modelo inicial sin optimizar

In [ ]:
dt_inicial = DecisionTreeClassifier(random_state=42)
dt_cv = cross_val_score(dt_inicial, X_train, y_train, cv=5, scoring='f1')
print(f"Decision Tree (sin optimizar) - F1 CV: {dt_cv.mean():.4f} ± {dt_cv.std():.4f}")

### Optimización de Hiperparámetros con GridSearchCV

In [ ]:
# Grid razonado:
# max_depth: evita sobreajuste, valores pequeños generalizan mejor
# min_samples_split/leaf: regularización
# class_weight: compensa el desbalanceo de clases
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'class_weight': ['balanced', None]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor F1 en CV:     {grid_search.best_score_:.4f}")

### Comparación de Modelos

In [ ]:
resultados = {
    'LR Baseline (sin opt.)': lr_cv.mean(),
    'Decision Tree (sin opt.)': dt_cv.mean(),
    'Decision Tree (optimizado)': grid_search.best_score_
}

print("Comparación de F1 en validación cruzada (5-fold):")
print("-" * 55)
for modelo, score in resultados.items():
    print(f"  {modelo:<35} F1: {score:.4f}")

plt.figure(figsize=(8, 4))
colors = ['coral', 'steelblue', 'mediumseagreen']
plt.bar(resultados.keys(), resultados.values(), color=colors, edgecolor='black')
plt.title('F1 en validación cruzada por modelo')
plt.ylabel('F1 Score')
plt.ylim(0, 0.8)
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()

### Evaluación final sobre Test

In [ ]:
best_dt = grid_search.best_estimator_
y_pred  = best_dt.predict(X_test)

print("=" * 65)
print("EVALUACIÓN FINAL - Árbol de Decisión Optimizado (TEST SET)")
print("=" * 65)
print(classification_report(y_test, y_pred, target_names=['No suscribe','Suscribe']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['No suscribe','Suscribe']).plot(cmap='Blues', ax=ax)
ax.set_title('Matriz de Confusión - Test Set')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables del árbol optimizado
feat_imp = pd.Series(best_dt.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
feat_imp.head(10).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 10 Features más importantes - Árbol Optimizado')
plt.ylabel('Importancia')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Top 5 features:", feat_imp.head().index.tolist())

### Conclusiones

El árbol de decisión optimizado mejora al baseline de regresión logística en F1.

**Hallazgos clave:**
- El dataset está muy desbalanceado (~88% "no"), por lo que el F1 es más informativo que el accuracy.
- La variable `duration` (duración del contacto) es la más predictiva: conversaciones largas indican mayor interés.
- El uso de `class_weight='balanced'` es fundamental para mejorar la detección de suscriptores.
- La optimización de hiperparámetros (`max_depth`, `min_samples_leaf`) evita el sobreajuste del árbol sin restricciones.
- El árbol optimizado equilibra mejor precisión y recall en la clase positiva.
